In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"
input_file = DATA_DIR / "mexico_city_covid_data_dates.parquet"

In [2]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"
input_file = DATA_DIR / "mexico_city_covid_data_dates.parquet"

df_lazy = pl.scan_parquet(input_file)

primer_registro = df_lazy.head(1).collect()

print("=== ANALISIS DE COLUMNAS, TIPOS Y VALORES ===")
for col in primer_registro.columns:
    tipo = primer_registro[col].dtype
    valor = primer_registro[col][0]
    print(f"Columna: {col:<18} | Tipo: {str(tipo):<10} | Valor de ejemplo: {valor}")

=== ANALISIS DE COLUMNAS, TIPOS Y VALORES ===
Columna: _id                | Tipo: Binary     | Valor de ejemplo: b'`:\x7f\xd9\x07\x7f\x00\x00'
Columna: ASMA               | Tipo: String     | Valor de ejemplo: 2
Columna: CARDIOVASCULAR     | Tipo: String     | Valor de ejemplo: 2
Columna: CLASIFICACION_FINAL_COVID | Tipo: String     | Valor de ejemplo: 7
Columna: CLASIFICACION_FINAL_FLU | Tipo: String     | Valor de ejemplo: 6
Columna: DIABETES           | Tipo: String     | Valor de ejemplo: 2
Columna: EDAD               | Tipo: Int64      | Valor de ejemplo: 51
Columna: EMBARAZO           | Tipo: String     | Valor de ejemplo: 97
Columna: ENTIDAD_NAC        | Tipo: String     | Valor de ejemplo: 09
Columna: ENTIDAD_RES        | Tipo: String     | Valor de ejemplo: 09
Columna: ENTIDAD_UM         | Tipo: String     | Valor de ejemplo: 09
Columna: EPOC               | Tipo: String     | Valor de ejemplo: 2
Columna: FECHA_DEF          | Tipo: Date       | Valor de ejemplo: None
Columna: 

Catalogo Oficial

In [2]:
import requests
from pathlib import Path
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

catalog_url = "https://datosabiertos.salud.gob.mx/gobmx/salud/datos_abiertos/diccionario_datos_abiertos.zip"


with requests.get(catalog_url, stream=True) as response:
  response.raise_for_status()

  filepath = DATA_DIR / "diccionario_datos_abiertos.zip"

  with filepath.open("wb") as file:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
      if chunk:
        file.write(chunk)

print(f"File saved at {filepath}")

import zipfile

path = DATA_DIR / "diccionario_datos_abiertos.zip"
with zipfile.ZipFile(path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR / "diccionario_datos_abiertos")


File saved at /home/netfoor/Documents/university/KDD/covid19/data/diccionario_datos_abiertos.zip


In [5]:
import pandas as pd

excel_file = pd.ExcelFile(DATA_DIR / "diccionario_datos_abiertos" / "240708 Catalogos.xlsx")

print("Sheets found in '240708 Catalogos.xlsx':")
print(excel_file.sheet_names)

Sheets found in '240708 Catalogos.xlsx':
['Catálogo ORIGEN', 'Catálogo SECTOR', 'Catálogo SEXO', 'Catálogo TIPO_PACIENTE', 'Catálogo SI_NO', 'Catálogo NACIONALIDAD', 'Catálogo RESULTADO_LAB', 'Catálogo RESULTADO_PCR', 'Catálogo RESULTADO_ANTIGENO', 'Cat CLASIFICACION_FINAL_COVID', 'Cat CLASIFICACION_FINAL_FLU', 'Catálogo de ENTIDADES', 'Catálogo MUNICIPIOS']


In [7]:
schema = pl.read_parquet_schema(DATA_DIR / "polars" / "mexico_city_covid_data_dates.parquet")
print(schema.keys())

odict_keys(['_id', 'ASMA', 'CARDIOVASCULAR', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU', 'DIABETES', 'EDAD', 'EMBARAZO', 'ENTIDAD_NAC', 'ENTIDAD_RES', 'ENTIDAD_UM', 'EPOC', 'FECHA_DEF', 'FECHA_INGRESO', 'FECHA_SINTOMAS', 'HABLA_LENGUA_INDIG', 'HIPERTENSION', 'INDIGENA', 'INMUSUPR', 'INTUBADO', 'MIGRANTE', 'MUNICIPIO_RES', 'NACIONALIDAD', 'NEUMONIA', 'OBESIDAD', 'ORIGEN', 'OTRA_COM', 'OTRO_CASO', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN', 'RENAL_CRONICA', 'RESULTADO_ANTIGENO', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION', 'SECTOR', 'SEXO', 'TABAQUISMO', 'TIPO_PACIENTE', 'TOMA_MUESTRA_ANTIGENO', 'TOMA_MUESTRA_LAB', 'UCI', 'YEAR_ORIGIN', 'DIAS_SINTOMAS', 'DIAS_ATENCION', 'DIAS_ENFERMO'])


In [8]:
descriptores = pd.read_excel( DATA_DIR / "diccionario_datos_abiertos" / "240708 Descriptores_.xlsx")

display(descriptores[['NOMBRE DE VARIABLE', 'FORMATO O FUENTE']])

,NOMBRE DE VARIABLE,FORMATO O FUENTE
0,FECHA_ACTUALIZACION,AAAA-MM-DD
1,ID_REGISTRO,TEXTO
2,ORIGEN,CATÁLOGO: ORIGEN ...
3,SECTOR,CATÁLOGO: SECTOR ...
4,ENTIDAD_UM,CATALÓGO: ENTIDADES
5,SEXO,CATÁLOGO: SEXO
6,ENTIDAD_NAC,CATALÓGO: ENTIDADES
7,ENTIDAD_RES,CATALÓGO: ENTIDADES
8,MUNICIPIO_RES,CATALÓGO: MUNICIPIOS
9,TIPO_PACIENTE,CATÁLOGO: TIPO_PACIENTE


In [ ]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

input_file = DATA_DIR / "polars" / "mexico_city_covid_data_dates.parquet"
excel_path = DATA_DIR / "diccionario_datos_abiertos" / "240708 Catalogos.xlsx"
output_csv = DATA_DIR / "mexico_city_covid_final_report.csv"


print("Cargando catálogos desde el archivo Excel...")


cat_entidades = pl.read_excel(excel_path, sheet_name="Catálogo de ENTIDADES").select([
    pl.col("CLAVE_ENTIDAD").cast(pl.String),
    pl.col("ENTIDAD_FEDERATIVA")
])

cat_sector = pl.read_excel(excel_path, sheet_name="Catálogo SECTOR").select([
    pl.col("CLAVE").cast(pl.String).alias("CLAVE_SECTOR"),
    pl.col("DESCRIPCIÓN").alias("DESCRIPCION_SECTOR")
])

cat_clasif_covid = pl.read_excel(excel_path, sheet_name="Cat CLASIFICACION_FINAL_COVID").select([
    pl.col("CLAVE").cast(pl.String).alias("CLAVE_COVID"),
    pl.col("DESCRIPCIÓN").alias("DESCRIPCION_COVID")
])


cat_si_no = {"1": "SÍ", "2": "NO", "97": "NO APLICA", "98": "SE IGNORA", "99": "NO ESPECIFICADO"}
cat_sexo = {"1": "MUJER", "2": "HOMBRE", "99": "NO ESPECIFICADO"}
cat_origen = {"1": "USMER", "2": "FUERA DE USMER", "99": "NO ESPECIFICADO"}
cat_paciente = {"1": "AMBULATORIO", "2": "HOSPITALIZADO", "99": "NO ESPECIFICADO"}

columnas_medicas = [
    "ASMA", "CARDIOVASCULAR", "DIABETES", "EPOC", "HIPERTENSION", "INMUSUPR", 
    "NEUMONIA", "OBESIDAD", "RENAL_CRONICA", "TABAQUISMO", "INTUBADO", "UCI", 
    "EMBARAZO", "HABLA_LENGUA_INDIG", "INDIGENA", "OTRA_COM", "OTRO_CASO", 
    "TOMA_MUESTRA_LAB", "TOMA_MUESTRA_ANTIGENO", "MIGRANTE"
]


print("Iniciando pipeline de mapeo masivo...")
df_lazy = pl.scan_parquet(input_file)

processed_df = df_lazy.with_columns([
    pl.col("SEXO").replace(cat_sexo, default=pl.col("SEXO")),
    pl.col("ORIGEN").replace(cat_origen, default=pl.col("ORIGEN")),
    pl.col("TIPO_PACIENTE").replace(cat_paciente, default=pl.col("TIPO_PACIENTE")),
    *[pl.col(col).replace(cat_si_no, default=pl.col(col)) for col in columnas_medicas]
]).drop(["_id"]) 


processed_df = (
    processed_df
    .join(cat_entidades, left_on="ENTIDAD_UM", right_on="CLAVE_ENTIDAD", how="left")
    .rename({"ENTIDAD_FEDERATIVA": "NOM_ENTIDAD_UM"})
    
    .join(cat_entidades, left_on="ENTIDAD_NAC", right_on="CLAVE_ENTIDAD", how="left")
    .rename({"ENTIDAD_FEDERATIVA": "NOM_ENTIDAD_NAC"})
    
    .join(cat_entidades, left_on="ENTIDAD_RES", right_on="CLAVE_ENTIDAD", how="left")
    .rename({"ENTIDAD_FEDERATIVA": "NOM_ENTIDAD_RES"})
)

processed_df = (
    processed_df
    .join(cat_sector, left_on="SECTOR", right_on="CLAVE_SECTOR", how="left")
    
    .join(cat_clasif_covid, left_on="CLASIFICACION_FINAL_COVID", right_on="CLAVE_COVID", how="left")
)

columnas_a_quitar = ["ENTIDAD_UM", "ENTIDAD_NAC", "ENTIDAD_RES", "SECTOR", "CLASIFICACION_FINAL_COVID"]
processed_df = processed_df.drop(columnas_a_quitar)


processed_df.sink_csv(output_csv)

print(f"Archivo generado: {output_csv}")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
excel_path = DATA_DIR / "diccionario_datos_abiertos" / "240708 Catalogos.xlsx"

sheets = [
    "Catálogo MUNICIPIOS",
    "Catálogo RESULTADO_LAB",
    "Catálogo RESULTADO_PCR",
    "Catálogo RESULTADO_ANTIGENO",
    "Cat CLASIFICACION_FINAL_FLU",
    "Catálogo NACIONALIDAD",
    "Catálogo SEXO",
    "Catálogo ORIGEN",
    "Catálogo TIPO_PACIENTE",
    "Catálogo SI_NO",
    "Catálogo de ENTIDADES",
    "Catálogo SECTOR",
    "Cat CLASIFICACION_FINAL_COVID",
]

for sheet in sheets:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    print(f"\n=== {sheet} ===")
    print(f"Columnas: {list(df.columns)}")
    print(df.head(5).to_string())
    print(f"... ({len(df)} filas total)")



=== Catálogo MUNICIPIOS ===
Columnas: ['CLAVE_ENTIDAD', 'CLAVE_MUNICIPIO', 'MUNICIPIO']
   CLAVE_ENTIDAD  CLAVE_MUNICIPIO       MUNICIPIO
0              1                1  AGUASCALIENTES
1              1                2        ASIENTOS
2              1                3        CALVILLO
3              1                4           COSÍO
4              1                5     JESÚS MARÍA
... (2501 filas total)

=== Catálogo RESULTADO_LAB ===
Columnas: ['CLAVE', 'DESCRIPCIÓN', 'Unnamed: 2', 'Esta variable dejó de ser incluída en el corte de base del día 2024-07-08, fue']
   CLAVE                   DESCRIPCIÓN  Unnamed: 2 Esta variable dejó de ser incluída en el corte de base del día 2024-07-08, fue
0      1         POSITIVO A SARS-COV-2         NaN        reemplazada por las variables RESULTADO_PCR Y RESULTADO_PCR_COINFECCION
1      2      NO POSITIVO A SARS-COV-2         NaN                                                                            NaN
2      3           RESULTADO PENDIE

In [10]:

from pathlib import Path
import polars as pl
import pandas as pd

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
excel_path = DATA_DIR / "diccionario_datos_abiertos" / "240708 Catalogos.xlsx"
input_file = DATA_DIR / "polars" / "mexico_city_covid_data_dates.parquet"

# Ver catalogo RESULTADO_PCR completo
cat_pcr = pd.read_excel(excel_path, sheet_name="Catálogo RESULTADO_PCR")
print("=== RESULTADO_PCR catalog completo ===")
print(cat_pcr.to_string())


=== RESULTADO_PCR catalog completo ===
    CLAVE                     DESCRIPCIÓN
0       1             INFLUENZA AH1N1 PMD
1       2                  INFLUENZA A H1
2       3                  INFLUENZA A H3
3       4                     INFLUENZA B
4       5                        NEGATIVO
5       6             MUESTRA NO ADECUADA
6       7                      ADENOVIRUS
7       8                 PARAINFLUENZA 1
8       9                 PARAINFLUENZA 2
9      10                 PARAINFLUENZA 3
10     11    VIRUS SINCICIAL RESPIRATORIO
11     13    INFLUENZA A NO SUBTIPIFICADA
12     14                  INFLUENZA A H5
13     17               MUESTRA RECHAZADA
14     20  VIRUS SINCICIAL RESPIRATORIO A
15     21  VIRUS SINCICIAL RESPIRATORIO B
16     22                     CORONA 229E
17     23                     CORONA OC43
18     24                     CORONA SARS
19     25                     CORONA NL63
20     26                     CORONA HKU1
21     27        MUESTRA QUE NO AMPLI

In [11]:

# Verificar valores únicos de columnas que faltan mapear + ver row de ENTIDADES
df = pl.scan_parquet(input_file)

cols_to_check = [
    "RESULTADO_PCR", "RESULTADO_PCR_COINFECCION", "RESULTADO_ANTIGENO",
    "CLASIFICACION_FINAL_FLU", "NACIONALIDAD", "MUNICIPIO_RES",
    "ENTIDAD_RES", "PAIS_ORIGEN", "PAIS_NACIONALIDAD"
]

for col in cols_to_check:
    vals = df.select(pl.col(col).unique()).collect()[col].to_list()
    vals_sorted = sorted([str(v) for v in vals])
    print(f"{col}: {vals_sorted}")

# Ver fila 36 de ENTIDADES (por si hay código especial)
cat_ent = pd.read_excel(excel_path, sheet_name="Catálogo de ENTIDADES")
print(f"\n=== ENTIDADES (todas) ===")
print(cat_ent.to_string())


RESULTADO_PCR: ['1', '10', '11', '13', '14', '17', '2', '20', '21', '22', '23', '25', '26', '27', '28', '29', '3', '32', '33', '34', '36', '37', '4', '5', '6', '7', '8', '9', '97', '997', '999']
RESULTADO_PCR_COINFECCION: ['1', '10', '11', '13', '17', '2', '20', '21', '22', '23', '25', '26', '27', '28', '29', '3', '32', '33', '34', '37', '4', '5', '6', '7', '8', '9', '997', '998', '999']
RESULTADO_ANTIGENO: ['1', '2', '97']
CLASIFICACION_FINAL_FLU: ['3', '4', '5', '6', '7']
NACIONALIDAD: ['1', '2']
MUNICIPIO_RES: ['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069

In [12]:

from pathlib import Path
import polars as pl
import pandas as pd

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
excel_path = DATA_DIR / "diccionario_datos_abiertos" / "240708 Catalogos.xlsx"
input_file = DATA_DIR / "polars" / "mexico_city_covid_data_dates.parquet"
output_csv  = DATA_DIR / "mexico_city_covid_final_report.csv"

def excel_to_dict(sheet, key_col, val_col, pad=None):
    df = pd.read_excel(excel_path, sheet_name=sheet)
    result = {}
    for _, row in df.iterrows():
        if pd.isna(row[key_col]):
            continue
        key = str(int(row[key_col]))
        if pad:
            key = key.zfill(pad)
        result[key] = row[val_col]
    return result

cat_si_no        = excel_to_dict("Catálogo SI_NO",        "CLAVE", "DESCRIPCIÓN")
cat_sexo         = excel_to_dict("Catálogo SEXO",         "CLAVE", "DESCRIPCIÓN")
cat_origen       = excel_to_dict("Catálogo ORIGEN",       "CLAVE", "DESCRIPCIÓN")
cat_tipo_pac     = excel_to_dict("Catálogo TIPO_PACIENTE","CLAVE", "DESCRIPCIÓN")
cat_nacionalidad = excel_to_dict("Catálogo NACIONALIDAD", "CLAVE", "DESCRIPCIÓN")
cat_antigeno     = excel_to_dict("Catálogo RESULTADO_ANTIGENO", "CLAVE", "DESCRIPCIÓN")
cat_sector       = excel_to_dict("Catálogo SECTOR",       "CLAVE", "DESCRIPCIÓN")
cat_pcr          = excel_to_dict("Catálogo RESULTADO_PCR","CLAVE", "DESCRIPCIÓN")
cat_pcr["97"]    = "NO APLICA (CASO SIN MUESTRA)"
cat_clasif_covid = excel_to_dict("Cat CLASIFICACION_FINAL_COVID", "CLAVE", "CLASIFICACIÓN")
cat_clasif_flu   = excel_to_dict("Cat CLASIFICACION_FINAL_FLU",   "CLAVE", "CLASIFICACIÓN")
cat_entidades    = excel_to_dict("Catálogo de ENTIDADES", "CLAVE_ENTIDAD", "ENTIDAD_FEDERATIVA", pad=2)

muni_df = pd.read_excel(excel_path, sheet_name="Catálogo MUNICIPIOS")
cat_municipios = {
    f"{int(r['CLAVE_ENTIDAD']):02d}_{int(r['CLAVE_MUNICIPIO']):03d}": r['MUNICIPIO']
    for _, r in muni_df.iterrows()
}

print("Catálogos cargados:")
print(f"  SI_NO        : {cat_si_no}")
print(f"  SEXO         : {cat_sexo}")
print(f"  ENTIDADES 09 : {cat_entidades.get('09')}")
print(f"  PCR (sample) : { {k: cat_pcr[k] for k in list(cat_pcr)[:4]} }")
print(f"  COVID clasif : {cat_clasif_covid}")
print(f"  FLU clasif   : {cat_clasif_flu}")
print(f"  Municipios CDMX 007: {cat_municipios.get('09_007')}")


Catálogos cargados:
  SI_NO        : {'1': 'SI ', '2': 'NO ', '97': 'NO APLICA', '98': 'SE IGNORA', '99': 'NO ESPECIFICADO'}
  SEXO         : {'1': 'MUJER', '2': 'HOMBRE', '99': 'NO ESPECIFICADO'}
  ENTIDADES 09 : CIUDAD DE MÉXICO
  PCR (sample) : {'1': 'INFLUENZA AH1N1 PMD', '2': 'INFLUENZA A H1', '3': 'INFLUENZA A H3', '4': 'INFLUENZA B'}
  COVID clasif : {'1': 'CASO DE COVID-19 CONFIRMADO POR ASOCIACIÓN CLÍNICA EPIDEMIOLÓGICA', '2': 'CASO DE COVID-19 CONFIRMADO POR COMITÉ DE  DICTAMINACIÓN', '3': 'CASO DE SARS-COV-2  CONFIRMADO', '4': 'INVÁLIDO POR LABORATORIO', '5': 'NO REALIZADO POR LABORATORIO', '6': 'CASO SOSPECHOSO', '7': 'NEGATIVO A SARS-COV-2'}
  FLU clasif   : {'3': 'CASO DE INFLUENZA CONFIRMADO', '4': 'INVÁLIDO POR LABORATORIO', '5': 'NO REALIZADO POR LABORATORIO', '6': 'CASO SOSPECHOSO', '7': 'NEGATIVO A INFLUENZA'}
  Municipios CDMX 007: IZTAPALAPA


In [13]:

COLS_SI_NO = [
    "ASMA", "CARDIOVASCULAR", "DIABETES", "EPOC", "HIPERTENSION", "INMUSUPR",
    "NEUMONIA", "OBESIDAD", "RENAL_CRONICA", "TABAQUISMO", "INTUBADO", "UCI",
    "EMBARAZO", "HABLA_LENGUA_INDIG", "INDIGENA", "OTRA_COM", "OTRO_CASO",
    "TOMA_MUESTRA_LAB", "TOMA_MUESTRA_ANTIGENO", "MIGRANTE",
]

df_lazy = pl.scan_parquet(input_file)

# Paso 1: crear columna MUNICIPIO antes de modificar ENTIDAD_RES
df_lazy = df_lazy.with_columns(
    pl.concat_str([pl.col("ENTIDAD_RES"), pl.lit("_"), pl.col("MUNICIPIO_RES")])
    .replace(cat_municipios, default=None)
    .alias("MUNICIPIO")
)

# Paso 2: reemplazar todos los catálogos in-place
df_lazy = df_lazy.with_columns([
    pl.col("SEXO")            .replace(cat_sexo,         default=pl.col("SEXO")),
    pl.col("ORIGEN")          .replace(cat_origen,       default=pl.col("ORIGEN")),
    pl.col("TIPO_PACIENTE")   .replace(cat_tipo_pac,     default=pl.col("TIPO_PACIENTE")),
    pl.col("NACIONALIDAD")    .replace(cat_nacionalidad, default=pl.col("NACIONALIDAD")),
    pl.col("PAIS_ORIGEN")     .replace({"97": "NO APLICA"}, default=pl.col("PAIS_ORIGEN")),
    pl.col("ENTIDAD_UM") .replace(cat_entidades, default=pl.col("ENTIDAD_UM")),
    pl.col("ENTIDAD_NAC").replace(cat_entidades, default=pl.col("ENTIDAD_NAC")),
    pl.col("ENTIDAD_RES").replace(cat_entidades, default=pl.col("ENTIDAD_RES")),
    pl.col("SECTOR").replace(cat_sector, default=pl.col("SECTOR")),
    pl.col("RESULTADO_ANTIGENO")        .replace(cat_antigeno, default=pl.col("RESULTADO_ANTIGENO")),
    pl.col("RESULTADO_PCR")             .replace(cat_pcr,      default=pl.col("RESULTADO_PCR")),
    pl.col("RESULTADO_PCR_COINFECCION") .replace(cat_pcr,      default=pl.col("RESULTADO_PCR_COINFECCION")),
    pl.col("CLASIFICACION_FINAL_COVID").replace(cat_clasif_covid, default=pl.col("CLASIFICACION_FINAL_COVID")),
    pl.col("CLASIFICACION_FINAL_FLU") .replace(cat_clasif_flu,   default=pl.col("CLASIFICACION_FINAL_FLU")),
    *[pl.col(c).replace(cat_si_no, default=pl.col(c)) for c in COLS_SI_NO],
])

# Paso 3: quitar columnas innecesarias
df_lazy = df_lazy.drop(["_id", "MUNICIPIO_RES"])

# Verificación
raw_row    = pl.scan_parquet(input_file).head(1).collect()
mapped_row = df_lazy.head(1).collect()

print("=== VERIFICACIÓN: CRUDO vs MAPEADO (primer registro) ===")
for col in mapped_row.columns:
    crudo  = raw_row[col][0] if col in raw_row.columns else "—"
    mapped = mapped_row[col][0]
    if str(crudo) != str(mapped):
        print(f"  {col:<42} {str(crudo):<12} → {mapped}")
    else:
        print(f"  {col:<42} {str(crudo):<12}   (sin cambio)")


=== VERIFICACIÓN: CRUDO vs MAPEADO (primer registro) ===
  ASMA                                       2            → NO 
  CARDIOVASCULAR                             2            → NO 
  CLASIFICACION_FINAL_COVID                  7            → NEGATIVO A SARS-COV-2
  CLASIFICACION_FINAL_FLU                    6            → CASO SOSPECHOSO
  DIABETES                                   2            → NO 
  EDAD                                       51             (sin cambio)
  EMBARAZO                                   97           → NO APLICA
  ENTIDAD_NAC                                09           → CIUDAD DE MÉXICO
  ENTIDAD_RES                                09           → CIUDAD DE MÉXICO
  ENTIDAD_UM                                 09           → CIUDAD DE MÉXICO
  EPOC                                       2            → NO 
  FECHA_DEF                                  None           (sin cambio)
  FECHA_INGRESO                              2020-08-18     (sin cambio)
  FECHA_S

/tmp/ipykernel_3823/2625838534.py:13: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  .replace(cat_municipios, default=None)
/tmp/ipykernel_3823/2625838534.py:19: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("SEXO")            .replace(cat_sexo,         default=pl.col("SEXO")),
/tmp/ipykernel_3823/2625838534.py:20: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("ORIGEN")          .replace(cat_origen,       default=pl.col("ORIGEN")),
/tmp/ipykernel_3823/2625838534.py:21: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while r

In [14]:

import warnings
warnings.filterwarnings("error", category=DeprecationWarning)

COLS_SI_NO = [
    "ASMA", "CARDIOVASCULAR", "DIABETES", "EPOC", "HIPERTENSION", "INMUSUPR",
    "NEUMONIA", "OBESIDAD", "RENAL_CRONICA", "TABAQUISMO", "INTUBADO", "UCI",
    "EMBARAZO", "HABLA_LENGUA_INDIG", "INDIGENA", "OTRA_COM", "OTRO_CASO",
    "TOMA_MUESTRA_LAB", "TOMA_MUESTRA_ANTIGENO", "MIGRANTE",
]

df_lazy = pl.scan_parquet(input_file)

df_lazy = df_lazy.with_columns(
    pl.concat_str([pl.col("ENTIDAD_RES"), pl.lit("_"), pl.col("MUNICIPIO_RES")])
    .replace_strict(cat_municipios, default=None)
    .alias("MUNICIPIO")
)

df_lazy = df_lazy.with_columns([
    pl.col("SEXO")          .replace(cat_sexo),
    pl.col("ORIGEN")        .replace(cat_origen),
    pl.col("TIPO_PACIENTE") .replace(cat_tipo_pac),
    pl.col("NACIONALIDAD")  .replace(cat_nacionalidad),
    pl.col("PAIS_ORIGEN")   .replace({"97": "NO APLICA"}),
    pl.col("ENTIDAD_UM") .replace(cat_entidades),
    pl.col("ENTIDAD_NAC").replace(cat_entidades),
    pl.col("ENTIDAD_RES").replace(cat_entidades),
    pl.col("SECTOR").replace(cat_sector),
    pl.col("RESULTADO_ANTIGENO")        .replace(cat_antigeno),
    pl.col("RESULTADO_PCR")             .replace(cat_pcr),
    pl.col("RESULTADO_PCR_COINFECCION") .replace(cat_pcr),
    pl.col("CLASIFICACION_FINAL_COVID").replace(cat_clasif_covid),
    pl.col("CLASIFICACION_FINAL_FLU") .replace(cat_clasif_flu),
    *[pl.col(c).replace(cat_si_no) for c in COLS_SI_NO],
])

df_lazy = df_lazy.drop(["_id", "MUNICIPIO_RES"])

raw_row    = pl.scan_parquet(input_file).head(1).collect()
mapped_row = df_lazy.head(1).collect()

print("=== VERIFICACIÓN (sin warnings): CRUDO → MAPEADO ===")
for col in mapped_row.columns:
    crudo  = raw_row[col][0] if col in raw_row.columns else "—"
    mapped = mapped_row[col][0]
    if str(crudo) != str(mapped):
        print(f"  {col:<42} {str(crudo):<12} → {mapped}")

warnings.resetwarnings()
print("\n¡Sin deprecation warnings!")


=== VERIFICACIÓN (sin warnings): CRUDO → MAPEADO ===
  ASMA                                       2            → NO 
  CARDIOVASCULAR                             2            → NO 
  CLASIFICACION_FINAL_COVID                  7            → NEGATIVO A SARS-COV-2
  CLASIFICACION_FINAL_FLU                    6            → CASO SOSPECHOSO
  DIABETES                                   2            → NO 
  EMBARAZO                                   97           → NO APLICA
  ENTIDAD_NAC                                09           → CIUDAD DE MÉXICO
  ENTIDAD_RES                                09           → CIUDAD DE MÉXICO
  ENTIDAD_UM                                 09           → CIUDAD DE MÉXICO
  EPOC                                       2            → NO 
  HABLA_LENGUA_INDIG                         2            → NO 
  HIPERTENSION                               1            → SI 
  INDIGENA                                   2            → NO 
  INMUSUPR                              

In [15]:

import time
print(f"Guardando CSV en:\n  {output_csv}\n")
t0 = time.time()
df_lazy.sink_csv(output_csv)
t1 = time.time()
print(f"Tiempo: {t1-t0:.1f}s")

# Verificar tamaño y primeras líneas
import subprocess
result = subprocess.run(["wc", "-l", str(output_csv)], capture_output=True, text=True)
print(f"Líneas en CSV: {result.stdout.strip()}")

size_mb = output_csv.stat().st_size / 1024 / 1024
print(f"Tamaño: {size_mb:.1f} MB")


Guardando CSV en:
  /home/netfoor/Documents/university/KDD/covid19/data/mexico_city_covid_final_report.csv

Tiempo: 28.2s
Líneas en CSV: 7840212 /home/netfoor/Documents/university/KDD/covid19/data/mexico_city_covid_final_report.csv
Tamaño: 2936.5 MB


In [16]:
import subprocess
from pathlib import Path

ROOT = Path.cwd().parent
csv_path     = ROOT / "data" / "mexico_city_covid_final_report.csv"
parquet_path = ROOT / "data" / "polars" / "mexico_city_covid_data_dates.parquet"

# Filas CSV (wc -l cuenta el header, restamos 1)
csv_lines = int(subprocess.run(["wc", "-l", str(csv_path)], capture_output=True, text=True).stdout.split()[0])
csv_rows  = csv_lines - 1

# Filas parquet
parquet_rows = pl.scan_parquet(parquet_path).select(pl.len()).collect()[0, 0]

# Tamaños
csv_mb     = csv_path.stat().st_size     / 1024 / 1024
parquet_mb = parquet_path.stat().st_size / 1024 / 1024

print(f"{'':30} {'FILAS':>12} {'TAMAÑO':>10}")
print("-" * 55)
print(f"{'Parquet (original)':30} {parquet_rows:>12,} {parquet_mb:>9.1f} MB")
print(f"{'CSV (mapeado)':30} {csv_rows:>12,} {csv_mb:>9.1f} MB")
print("-" * 55)
print(f"{'Diferencia de filas':30} {csv_rows - parquet_rows:>12,}")
print(f"{'Factor de tamaño CSV/parquet':30} {csv_mb / parquet_mb:>11.1f}x")

                                      FILAS     TAMAÑO
-------------------------------------------------------
Parquet (original)                7,840,211      62.8 MB
CSV (mapeado)                     7,840,211    2936.5 MB
-------------------------------------------------------
Diferencia de filas                       0
Factor de tamaño CSV/parquet          46.8x
